In [ ]:
%config InlineBackend.figure_format = 'svg'

import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

# System parameters
m_L = 80000.0
m_C = 40000.0
b_CL = 50000.0
b_L = 5000.0
b_C = 3000.0
k_CL = 1250000.0 / 3.0
v_cruise = 100.0 / 3.6
g = 9.81

# Plant transfer function G_CL(s) = delta(s) / F_CL(s)
num_plant = np.array([m_L + m_C, b_L + b_C]) / (m_L * m_C)
den_plant = np.array([
    1.0,
    (m_L * b_C + m_C * b_L + b_CL * (m_L + m_C)) / (m_L * m_C),
    (b_L * b_C + b_CL * (b_L + b_C) + k_CL * (m_L + m_C)) / (m_L * m_C),
    k_CL * (b_L + b_C) / (m_L * m_C)
])

# Lag compensator C(s) = K * (s + z) / (s + p)
K = 1.6e6
z = 0.5
p = 0.02

num_comp = np.array([K, K * z])
den_comp = np.array([1.0, p])

# Loop transfer function L(s) = C(s) * G_CL(s)
num_loop = np.polymul(num_comp, num_plant)
den_loop = np.polymul(den_comp, den_plant)

w = np.logspace(-2, 2, 1000)
_, mag_plant, phase_plant = signal.bode((num_plant, den_plant), w)
_, mag_loop, phase_loop = signal.bode((num_loop, den_loop), w)

idx_cross = np.argmin(np.abs(mag_loop))
w_c = w[idx_cross]
pm = 180.0 + phase_loop[idx_cross]

L_0 = (K * z / p) * (1.0 / k_CL)
print(f'DC Loop Gain L(0): {L_0:.1f} ({20*np.log10(L_0):.2f} dB)')
print(f'Loop Gain at w=0.01 rad/s: {10**(mag_loop[0]/20):.1f} ({mag_loop[0]:.2f} dB)')
print(f'Gain Crossover Frequency: {w_c:.2f} rad/s')
print(f'Phase Margin: {pm:.2f} deg')

# Bode Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.5), sharex=True)
ax1.semilogx(w, mag_plant, 'b--', label=r'Plant $G_{CL}(s)$')
ax1.semilogx(w, mag_loop, 'r-', linewidth=1.5, label=r'Compensated $L(s)$')
ax1.axhline(0, color='gray', linestyle=':', alpha=0.7)
ax1.axvline(w_c, color='k', linestyle=':', alpha=0.5)
ax1.set_ylabel('Magnitude (dB)')
ax1.set_title('Bode Magnitude Response')
ax1.grid(True, which='both', linestyle='--', alpha=0.5)
ax1.legend(loc='lower left')

ax2.semilogx(w, phase_plant, 'b--', label=r'Plant $G_{CL}(s)$')
ax2.semilogx(w, phase_loop, 'r-', linewidth=1.5, label=r'Compensated $L(s)$')
ax2.axhline(-180, color='gray', linestyle=':', alpha=0.7)
ax2.axvline(w_c, color='k', linestyle=':', alpha=0.5)
ax2.set_ylabel('Phase (deg)')
ax2.set_xlabel('Frequency (rad/s)')
ax2.set_title('Bode Phase Response')
ax2.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('BodePlot.pdf')
plt.show()

# Simulation Setup (RK4)
# Simulation time step derived from natural frequency omega_n
omega_n = 3.957  # rad/s
T_n = 2 * np.pi / omega_n  # approx 1.588 s
dt = 0.001  # dt <= T_n / 1000 for accuracy and stability
time = np.arange(0, 15.0, dt)
N = len(time)
dFL = 50000.0

def closed_loop_deriv(state, t):
    vL, vC, delta, xc = state
    F_CL = -K * (delta + (z - p) * xc)
    F_spring = k_CL * (0.2 + delta)
    F_damper = b_CL * (vL - vC)
    F_drag_L = b_L * vL
    F_drag_C = b_C * vC
    F_traction = (b_L + b_C) * v_cruise + (dFL if t >= 0 else 0.0)
    
    aL = (F_traction + F_CL - F_spring - F_damper - F_drag_L) / m_L
    aC = (F_spring + F_damper - F_CL - F_drag_C) / m_C
    delta_dot = vL - vC
    xc_dot = -p * xc + delta
    return np.array([aL, aC, delta_dot, xc_dot]), F_CL, aC

def open_loop_deriv(state, t):
    vL, vC, delta = state
    F_spring = k_CL * (0.2 + delta)
    F_damper = b_CL * (vL - vC)
    F_drag_L = b_L * vL
    F_drag_C = b_C * vC
    F_traction = (b_L + b_C) * v_cruise + (dFL if t >= 0 else 0.0)
    
    aL = (F_traction - F_spring - F_damper - F_drag_L) / m_L
    aC = (F_spring + F_damper - F_drag_C) / m_C
    delta_dot = vL - vC
    return np.array([aL, aC, delta_dot]), aC

# Open Loop Integration
x_ol = np.array([v_cruise, v_cruise, 0.0])
hist_ol = np.zeros((N, 3))
for i, t in enumerate(time):
    hist_ol[i] = x_ol
    k1, _ = open_loop_deriv(x_ol, t)
    k2, _ = open_loop_deriv(x_ol + 0.5 * dt * k1, t + 0.5 * dt)
    k3, _ = open_loop_deriv(x_ol + 0.5 * dt * k2, t + 0.5 * dt)
    k4, _ = open_loop_deriv(x_ol + dt * k3, t + dt)
    x_ol = x_ol + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

# Closed Loop Integration
x_cl = np.array([v_cruise, v_cruise, 0.0, 0.0])
hist_cl = np.zeros((N, 4))
F_CL_hist = np.zeros(N)
aC_cl_hist = np.zeros(N)

for i, t in enumerate(time):
    hist_cl[i] = x_cl
    deriv, F_CL_now, aC_now = closed_loop_deriv(x_cl, t)
    F_CL_hist[i] = F_CL_now
    aC_cl_hist[i] = aC_now
    
    k1, _, _ = closed_loop_deriv(x_cl, t)
    k2, _, _ = closed_loop_deriv(x_cl + 0.5 * dt * k1, t + 0.5 * dt)
    k3, _, _ = closed_loop_deriv(x_cl + 0.5 * dt * k2, t + 0.5 * dt)
    k4, _, _ = closed_loop_deriv(x_cl + dt * k3, t + dt)
    x_cl = x_cl + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

delta_ol_mm = hist_ol[:, 2] * 1000.0
delta_cl_mm = hist_cl[:, 2] * 1000.0
F_CL_kN = F_CL_hist / 1000.0
aC_g = aC_cl_hist / g

# Verification Figures
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(7.5, 6.0), sharex=True)

ax1.plot(time, delta_ol_mm, 'k--', label=r'Uncompensated ($\delta_{OL}$)', alpha=0.7)
ax1.plot(time, delta_cl_mm, 'b-', linewidth=1.6, label=r'Compensated ($\delta_{CL}$)')
ax1.axhline(15, color='r', linestyle=':', label='Peak Limit (+/-15 mm)')
ax1.axhline(-15, color='r', linestyle=':')
ax1.axhline(5, color='g', linestyle='-.', label='Settling Band (+/-5 mm)')
ax1.axhline(-5, color='g', linestyle='-.')
ax1.axvline(2.0, color='orange', linestyle='--', alpha=0.7, label='Settling Time (2.0 s)')
ax1.set_ylabel(r'$\delta(t)$ (mm)')
ax1.set_title('Coupler Deflection Response')
ax1.set_xlim(0, 15)
ax1.set_ylim(-10, 65)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right', fontsize=8, ncol=2)

ax2.plot(time, F_CL_kN, 'm-', linewidth=1.5, label=r'Actuator Force $F_{CL}(t)$')
ax2.axhline(25, color='r', linestyle=':', label='Limit (+/-25 kN)')
ax2.axhline(-25, color='r', linestyle=':')
ax2.set_ylabel(r'$F_{CL}$ (kN)')
ax2.set_title('Linear Actuator Force')
ax2.set_xlim(0, 15)
ax2.set_ylim(-30, 10)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='lower right', fontsize=8)

ax3.plot(time, aC_g, color='purple', linewidth=1.5, label=r'Cabin Acceleration $a_C(t)$')
ax3.axhline(0.06, color='r', linestyle=':', label='Peak Limit (+/-0.06 g)')
ax3.axhline(-0.06, color='r', linestyle=':')
ax3.axhline(0.045, color='darkorange', linestyle='-.', label='Comfort Band (+/-0.045 g)')
ax3.axhline(-0.045, color='darkorange', linestyle='-.')
ax3.set_ylabel(r'$a_C$ (g)')
ax3.set_xlabel('Time (s)')
ax3.set_title('Cabin Acceleration')
ax3.set_xlim(0, 15)
ax3.set_ylim(-0.08, 0.14)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.legend(loc='upper right', fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('ClosedLoopResponse.pdf')
plt.show()

idx_2s = int(2.0 / dt)
print(f'Peak Deflection: {np.max(delta_cl_mm):.2f} mm')
print(f'Deflection at 2s: {delta_cl_mm[idx_2s]:.2f} mm')
delta_ss_th = (9e-7 * dFL) / (1.0 + L_0) * 1000.0
print(f'Deflection at t=15s: {delta_cl_mm[-1]:.2f} mm (Theoretical steady-state: {delta_ss_th:.2f} mm)')
print(f'Peak Actuator Force: {np.max(np.abs(F_CL_kN)):.2f} kN')
print(f'Peak Cabin Acceleration: {np.max(np.abs(aC_g)):.3f} g')


DC Loop Gain L(0): 96.0 (39.65 dB)
Loop Gain at w=0.01 rad/s: 85.9 (38.68 dB)
Gain Crossover Frequency: 8.53 rad/s
Phase Margin: 12.98 deg
Peak Deflection: 13.64 mm
Deflection at 2s: 4.18 mm
Deflection at t=15s: 0.54 mm (Theoretical steady-state: 0.46 mm)
Peak Actuator Force: 23.80 kN
Peak Cabin Acceleration: 0.074 g
